<div style="display: flex; align-items: center; padding: 20px; background-color: #f0f2f6; border-radius: 10px; border: 2px solid #007bff;">
    <img src="../logo.png" style="width: 80px; height: auto; margin-right: 20px;">
    <div style="flex: 1; text-align: left;">
    <h1 style="color: #007bff; margin-bottom: 5px;">GLY 6739.017S26: Computational Seismology</h1>
    <h3 style="color: #666;">Notebook 10: Instrument Responses (Plotting only)</h3>
    <p style="color: red;"><i>Glenn Thompson | Spring 2026</i></p>
    </div>
</div>

This notebook downloads **nominal** instrument responses from the **EarthScope/IRIS Nominal Response Library (NRL)** for a Nanometrics Trillium & Centaur combination, using ObsPy, and then shows:

1) **Sensor response** (frequency-domain + time-domain equivalent filter)  
2) **Digitizer response** (frequency-domain + time-domain equivalent filter)  
3) **Combined response** (frequency-domain + time-domain equivalent filter)

It also shows the response of a Raspberry Shake seismometer.

Notes:
- Frequency-domain plots use ObsPy's built-in `Inventory.plot_response()` (robust across versions).

## 1) Download responses from the NRL

We use two approaches:

- **Combined response**: public API  
  `nrl.get_response(datalogger_keys=..., sensor_keys=...)`

- **Sensor-only / Digitizer-only**: internal helper (simplest way to get them separately)  
  `nrl._get_response("sensors", keys=...)` and `nrl._get_response("dataloggers", keys=...)`

In ObsPy, `_get_response()` returns a `Response` object directly.

In [ ]:
from obspy.clients.nrl import NRL

nrl = NRL()

sensor_keys = ["Nanometrics", "Trillium Compact 120 (Vault, Posthole, OBS)", "754 V/m/s"]
datalogger_keys = ["Nanometrics", "Centaur", "40 Vpp (1)", "Off", "Linear phase", "100"]

resp_combined = nrl.get_response(datalogger_keys=datalogger_keys, sensor_keys=sensor_keys)
resp_sensor, _ = nrl._get_response("sensors", keys=sensor_keys)
resp_digitizer, _ = nrl._get_response("dataloggers", keys=datalogger_keys)
print("Combined response:", resp_combined)


## 2) Wrap each `Response` in an `Inventory`

In [ ]:
from obspy.core.inventory import Inventory, Network, Station, Channel, Site, Response
from obspy import UTCDateTime

def response_to_inventory(resp: Response, net="XX", sta="NRL1", loc="00", cha="HHZ", sr=100.0) -> Inventory:
    if not isinstance(resp, Response):
        raise TypeError(f"response_to_inventory expected obspy Response, got {type(resp)!r}")

    neto = Network(code=net)
    stao = Station(
        code=sta,
        latitude=0, longitude=0, elevation=0,
        creation_date=UTCDateTime(2020, 1, 1),
        site=Site(name="NRL"),
    )
    chao = Channel(
        code=cha,
        location_code=loc,
        latitude=0, longitude=0,
        elevation=0, depth=0,
        azimuth=0, dip=-90,
        sample_rate=sr,
    )
    chao.response = resp
    stao.channels.append(chao)
    neto.stations.append(stao)

    return Inventory(networks=[neto], source="NRL via ObsPy")

inv_sensor    = response_to_inventory(resp_sensor,    sta="SENSOR",   sr=100.0)
inv_digitizer = response_to_inventory(resp_digitizer, sta="DIGI",     sr=100.0)
inv_combined  = response_to_inventory(resp_combined,  sta="COMBINED", sr=100.0)

## 3) Frequency-domain response plots (magnitude + phase)

`Inventory.plot_response()` is the most robust way to plot response magnitude/phase, and it expects an Inventory/Channel.
So we build a tiny "dummy" Inventory for each response.

We plot with `output="VEL"` to match your combined response, which is:

**From M/S (ground velocity) to COUNTS**

In [ ]:
inv_sensor.plot_response(min_freq=0.001, output="VEL");
inv_digitizer.plot_response(min_freq=0.001, output="VEL");
inv_combined.plot_response(min_freq=0.001, output="VEL");

In [ ]:
inv_combined.plot_response(min_freq=0.001, output="DISP");

## Choose a Raspberry Shake, and download & plot the response

RCD29 is one of my Raspberry Shake & Boom instruments.
* Shake: a geophone (high-frequency short-period seismometer, peaks at 4.5 Hz)
* Boom: an infrasound sensor

In [ ]:
from obspy.clients.fdsn import Client
from obspy import UTCDateTime

rs = Client("RASPISHAKE")  # or Client("https://data.raspberryshake.org")
inv = rs.get_stations(network="AM", station="RCD29", channel='EHZ', level="response")
inv.plot_response(min_freq=0.001, output="VEL");

net = inv.select(network="AM", station="RCD29", channel="EHZ")[0]
sta = net.stations[0]
cha = sta.channels[0]
resp = cha.response
for i, stage in enumerate(resp.response_stages, start=1):
    print(f"Stage {i}: {type(stage).__name__}, gain={getattr(stage, 'stage_gain', None)}")



## Synthetic demo: deconvolution as *division* in the frequency domain

Below we simulate a near-delta pulse (a **Ricker wavelet**) as the "true" ground motion, apply an **instrument response** in the frequency domain,
and then "reconstitute" the signal by dividing by the frequency response (with a waterlevel to avoid blowing up noise where the response is tiny).

**Key identity** (frequency domain):

\[ X_{\text{obs}}(f) = H(f)\,X_{\text{true}}(f) \qquad\Rightarrow\qquad X_{\text{true}}(f) = \frac{X_{\text{obs}}(f)}{H(f)} \]

If you ran earlier cells that computed `resp_combined` (ObsPy `Response`) and/or created `inv_combined`, this section will use them automatically.
Otherwise it falls back to a simple toy bandpass response so the demo still runs.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from obspy import Trace, UTCDateTime

def ricker(t, f0):
    """Ricker wavelet centered at t=0 (a near-delta pulse), dominant frequency f0 (Hz)."""
    a = (np.pi * f0 * t)**2
    return (1.0 - 2.0*a) * np.exp(-a)

def amplitude_spectrum(x, dt):
    """One-sided amplitude spectrum."""
    n = len(x)
    X = np.fft.rfft(x)
    f = np.fft.rfftfreq(n, d=dt)
    A = np.abs(X)
    return f, A, X

def _eval_H_from_response(resp, dt, nfft, output="VEL"):
    """Evaluate complex frequency response H(f) for an ObsPy Response via evalresp."""
    f = np.fft.rfftfreq(nfft, d=dt)
    try:
        freqs, H = resp.get_evalresp_response(t_samp=dt, nfft=nfft, output=output)
    except TypeError:
        freqs, H = resp.get_evalresp_response(t_samp=dt, nfft=nfft)
    H = np.asarray(H)
    if H.shape[0] == nfft:
        H = H[:len(f)]
    if freqs is not None:
        freqs = np.asarray(freqs)
        if freqs.shape[0] == nfft:
            freqs = freqs[:len(f)]
        if freqs.shape == f.shape:
            f = freqs
    return f, H

def toy_bandpass_H(f, f1=0.2, f2=30.0, order=4):
    """Simple smooth bandpass magnitude with zero phase (toy model)."""
    hp = 1.0 / np.sqrt(1.0 + (f1/np.maximum(f, 1e-12))**(2*order))
    lp = 1.0 / np.sqrt(1.0 + (np.maximum(f, 1e-12)/f2)**(2*order))
    return hp * lp + 0j


### 1) Simulate a "true" waveform: Ricker wavelet + a touch of noise

We build an ObsPy `Trace` so it feels like real workflow.


In [ ]:
# Simulation parameters
sr = 100.0
dt = 1.0/sr
n = 4096
t = (np.arange(n) - n//2) * dt

# Near-delta pulse (Ricker)
f0 = 5.0  # dominant frequency (Hz)
x_true = ricker(t, f0)

# Add a little broadband noise (so you can *see* why waterlevel matters)
rng = np.random.default_rng(0)
x_true_noisy = x_true + 0.002 * rng.standard_normal(n)

tr_true = Trace(data=x_true_noisy.astype(np.float64))
tr_true.stats.starttime = UTCDateTime(2020, 1, 1)
tr_true.stats.sampling_rate = sr
tr_true.stats.channel = "HHZ"

plt.figure()
plt.plot(t, x_true_noisy)
plt.xlim(-2, 2)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Synthetic 'true' signal (Ricker wavelet + small noise)")
plt.show()


## Synthetic demo (ObsPy style): apply response (convolution) and recover with `remove_response()`

Here we do exactly what you described:

1. Create a **Ricker** wavelet as the *true* ground motion (we'll treat it as **velocity**, e.g., m/s).
2. **Apply the Trillium+Centaur combined response** (i.e., convolve in time ⇔ multiply by \(H(f)\) in frequency) to generate a synthetic **observed** seismogram in *counts*.
3. Use **`Trace.remove_response()`** with the same response to try to recover the original ground motion.
4. Plot time series + spectra.

Notes:
- `remove_response()` is a *regularized* deconvolution (water level, optional pre-filter), so recovery won't be perfect unless SNR is high and the pre-filter matches the usable band.
- This section assumes you already have `resp_combined` and `inv_combined` from earlier NRL cells in this notebook.


In [ ]:
from obspy.core.inventory import Inventory
from obspy.signal.util import _npts2nfft

# --- Guard rails: make sure the NRL response/inventory exist ---
if 'resp_combined' not in globals():
    raise RuntimeError("Expected `resp_combined` (ObsPy Response) from earlier NRL cells.")
if 'inv_combined' not in globals():
    raise RuntimeError("Expected `inv_combined` (Inventory) from earlier response_to_inventory() cells.")

assert isinstance(inv_combined, Inventory)


### 1) Define a 'true' ground-velocity Trace (Ricker wavelet)

We center the wavelet, and store it in an ObsPy `Trace` with a sampling rate.


In [ ]:
# Parameters (reuse if already defined above)
sr = float(globals().get('sr', 100.0))
dt = 1.0 / sr
n = int(globals().get('n', 4096))
t = (np.arange(n) - n//2) * dt

f0 = 5.0  # Hz dominant frequency
x_vel_true = ricker(t, f0).astype(np.float64)

# Optional: add a small amount of noise
rng = np.random.default_rng(1)
x_vel_true = (x_vel_true + 0.001 * rng.standard_normal(n)) * 1e-6 # realistic signal amplitude in m/s 

tr_vel_true = Trace(x_vel_true.copy())
tr_vel_true.stats.starttime = UTCDateTime(2020, 1, 1)
tr_vel_true.stats.sampling_rate = sr
tr_vel_true.stats.network = "XX"
tr_vel_true.stats.station = "COMBINED"
tr_vel_true.stats.location = "00"
tr_vel_true.stats.channel = "HHZ"

plt.figure()
plt.plot(t, tr_vel_true.data)
plt.xlim(-2, 2)
plt.xlabel("Time (s)")
plt.ylabel("Velocity (m/s)")
plt.title("True ground motion (velocity): Ricker wavelet" )
plt.show()


### 2) Apply the response to get a synthetic observed seismogram (counts)

In frequency domain:

\[ X_{\text{counts}}(f) = H_{\text{VEL}}(f)\,X_{\text{VEL}}(f) \]

where \(H_{\text{VEL}}(f)\) is the complex response mapping **velocity → counts**.


In [ ]:
# Choose FFT length (power-of-two-ish) for clean convolution
nfft = _npts2nfft(n)
f = np.fft.rfftfreq(nfft, d=dt)

# Evaluate complex response for velocity -> counts
# output="VEL" tells evalresp to use a velocity input when computing sensitivity/response.
# (i.e., response for VEL ground motion to COUNTS output)
fH, H_vel = _eval_H_from_response(resp_combined, dt=dt, nfft=nfft, output="VEL")
if fH.shape == f.shape:
    f = fH

X_vel = np.fft.rfft(tr_vel_true.data, n=nfft)
X_counts = H_vel * X_vel
x_counts_obs = np.fft.irfft(X_counts, n=nfft)[:n]

tr_counts_obs = tr_vel_true.copy()
tr_counts_obs.data = x_counts_obs.astype(np.float64) * 3.0172e8 # scale by 3e8 to get realistic counts amplitude (assuming ~3e8 V/(m/s) sensitivity)

# Attach response so remove_response can use it
tr_counts_obs.attach_response(inv_combined)

plt.figure()
plt.plot(t, tr_counts_obs.data)
plt.xlim(-2, 2)
plt.xlabel("Time (s)")
plt.ylabel("Counts (synthetic)")
plt.title("Observed seismogram (synthetic): true × response" )
plt.show()


### 3) Compare spectra: true velocity vs observed counts

This shows how the instrument shapes the spectrum (i.e., convolution in time).


In [ ]:
# Amplitude spectra
f_true, A_true, _ = amplitude_spectrum(tr_vel_true.data, dt)
f_obs,  A_obs,  _ = amplitude_spectrum(tr_counts_obs.data, dt)

# Also plot |H| scaled for visibility
Hmag = np.abs(H_vel)
Hmag_scaled = Hmag / (Hmag.max() if Hmag.max() > 0 else 1.0) * A_obs.max()

plt.figure()
plt.loglog(f_true[1:], A_true[1:], label="|X_true| (velocity)")
plt.loglog(f_obs[1:],  A_obs[1:],  label="|X_obs| (counts)")
plt.loglog(f[1:], Hmag_scaled[1:], label="|H_vel| (scaled)", linestyle="--")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplitude (arb.)")
plt.title("Spectra: applying the response (×H)" )
plt.legend()
plt.show()


### 4) Recover ground motion using ObsPy `remove_response()`

We'll ask for `output="VEL"` (velocity). Use a pre-filter that brackets the usable band
(and avoids huge amplification where the response is tiny).


In [ ]:
# Pick a sensible pre-filter based on your sample rate and typical broadband response.
# Adjust these corners to taste / to match the NRL response band.
pre_filt = (0.05, 0.1, 40.0, 45.0) if sr >= 100 else (0.02, 0.05, 0.4*sr, 0.45*sr)

tr_vel_rec = tr_counts_obs.copy()
tr_vel_rec.remove_response(
    inventory=inv_combined,
    output="VEL",
    pre_filt=pre_filt,
    water_level=60,   # dB water level (common default-ish)
    plot=False
)

plt.figure()
plt.plot(t, tr_vel_true.data, label="true vel", alpha=0.9)
plt.plot(t, tr_vel_rec.data,  label="recovered vel (remove_response)", alpha=0.8)
plt.xlim(-2, 2)
plt.xlabel("Time (s)")
plt.ylabel("Velocity (arb. units)")
plt.title("Time domain: true vs recovered" )
plt.legend()
plt.show()


### 5) Spectra after recovery

If things are behaving, the recovered spectrum should match the true spectrum in-band,
and diverge out-of-band (because of the regularization + pre-filter).


In [ ]:
f_rec, A_rec, _ = amplitude_spectrum(tr_vel_rec.data, dt)

plt.figure()
plt.loglog(f_true[1:], A_true[1:], label="|X_true| (velocity)")
plt.loglog(f_rec[1:],  A_rec[1:],  label="|X_rec| (velocity)")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplitude (arb.)")
plt.title("Spectra: after remove_response (≈ ÷H in-band)" )
plt.legend()
plt.show()
